# ReAct SQL Agent

Ask natural-language questions against the local `orders` table in SQLite.

OpenRouter model IDs must include the provider prefix, for example `openai/gpt-4o-mini` or `anthropic/claude-sonnet-4`.

In [15]:
import os
import sys
from pathlib import Path

from dotenv import load_dotenv
from langchain_community.agent_toolkits import create_sql_agent
from langchain_community.utilities import SQLDatabase
from langchain_openrouter import ChatOpenRouter

env_path = next(
    (path / ".env" for path in (Path.cwd(), *Path.cwd().parents) if (path / ".env").exists()),
    Path.cwd() / ".env",
)
load_dotenv(env_path, override=True)

if not os.getenv("OPENROUTER_API_KEY"):
    raise ValueError("OPENROUTER_API_KEY not found")

part3_dir = Path.cwd()
if not (part3_dir / "init_db.py").exists() and (part3_dir / "Part_3" / "init_db.py").exists():
    part3_dir = part3_dir / "Part_3"
if str(part3_dir) not in sys.path:
    sys.path.insert(0, str(part3_dir))

import init_db

database_path = init_db.init_db()
llm = ChatOpenRouter(model="openai/gpt-4o-mini", temperature=0, max_tokens=700)

db = SQLDatabase.from_uri(f"sqlite:///{database_path}")
sql_agent = create_sql_agent(
    llm,
    db=db,
    agent_type="tool-calling",
    verbose=True,
)

print("Database:", database_path)
print("Tables:", db.get_usable_table_names())

Database: /Users/cyberaan/Desktop/LangChange/Part_3/SalesDB/sales.db
Tables: ['orders']


In [16]:
question = "Who spent the most money, and how much did they spend?"

result = sql_agent.invoke({"input": question})
result["output"]



> Entering new SQL Agent Executor chain...

Invoking: `sql_db_list_tables` with `{}`


orders
Invoking: `sql_db_schema` with `{'table_names': 'orders'}`



CREATE TABLE orders (
	id INTEGER, 
	customer_name TEXT NOT NULL, 
	product_name TEXT NOT NULL, 
	quantity INTEGER NOT NULL, 
	price REAL NOT NULL, 
	total REAL NOT NULL, 
	PRIMARY KEY (id)
)

/*
3 rows from orders table:
id	customer_name	product_name	quantity	price	total
1	John Doe	Laptop	1	1000.0	1000.0
2	Jane Smith	Smartphone	2	500.0	1000.0
3	Bob Johnson	Tablet	3	200.0	600.0
*/
Invoking: `sql_db_query_checker` with `{'query': 'SELECT customer_name, SUM(total) as total_spent FROM orders GROUP BY customer_name ORDER BY total_spent DESC LIMIT 1;'}`


```sql
SELECT customer_name, SUM(total) as total_spent FROM orders GROUP BY customer_name ORDER BY total_spent DESC LIMIT 1;
```
Invoking: `sql_db_query` with `{'query': 'SELECT customer_name, SUM(total) as total_spent FROM orders GROUP BY customer_name ORDER BY total_spent DESC LIMIT

'The person who spent the most money is John Doe, and he spent $1000.00.'

In [17]:
follow_up = "What is the average order total?"

sql_agent.invoke({"input": follow_up})["output"]



> Entering new SQL Agent Executor chain...

Invoking: `sql_db_list_tables` with `{}`


orders
Invoking: `sql_db_schema` with `{'table_names': 'orders'}`



CREATE TABLE orders (
	id INTEGER, 
	customer_name TEXT NOT NULL, 
	product_name TEXT NOT NULL, 
	quantity INTEGER NOT NULL, 
	price REAL NOT NULL, 
	total REAL NOT NULL, 
	PRIMARY KEY (id)
)

/*
3 rows from orders table:
id	customer_name	product_name	quantity	price	total
1	John Doe	Laptop	1	1000.0	1000.0
2	Jane Smith	Smartphone	2	500.0	1000.0
3	Bob Johnson	Tablet	3	200.0	600.0
*/
Invoking: `sql_db_query_checker` with `{'query': 'SELECT AVG(total) AS average_order_total FROM orders;'}`


```sql
SELECT AVG(total) AS average_order_total FROM orders;
```
Invoking: `sql_db_query` with `{'query': 'SELECT AVG(total) AS average_order_total FROM orders;'}`


[(866.6666666666666,)]The average order total is approximately 866.67.

> Finished chain.


'The average order total is approximately 866.67.'

## ReAct agent with `SQLDatabaseToolkit`

Same database and LLM as above, but built manually with LangGraph's agent API.

In [18]:
from langchain.agents import create_agent
from langchain_community.agent_toolkits.sql.toolkit import SQLDatabaseToolkit

toolkit = SQLDatabaseToolkit(db=db, llm=llm)
sql_tools = toolkit.get_tools()

sql_react_agent = create_agent(llm, tools=sql_tools)
[tool.name for tool in sql_tools]

['sql_db_query', 'sql_db_schema', 'sql_db_list_tables', 'sql_db_query_checker']

In [19]:
example_query = "How much total sales we made for Tablet"

for event in sql_react_agent.stream(
    {"messages": [("user", example_query)]},
    stream_mode="values",
):
    event["messages"][-1].pretty_print()

================================ Human Message =================================

How much total sales we made for Tablet
================================== Ai Message ==================================
Tool Calls:
  sql_db_list_tables (call_Uub7my8gvgbEgEVEDOVBJigI)
 Call ID: call_Uub7my8gvgbEgEVEDOVBJigI
  Args:
================================= Tool Message =================================
Name: sql_db_list_tables

orders
================================== Ai Message ==================================
Tool Calls:
  sql_db_schema (call_usoTeyHFFBMJh6oyJRVToEI4)
 Call ID: call_usoTeyHFFBMJh6oyJRVToEI4
  Args:
    table_names: orders
================================= Tool Message =================================
Name: sql_db_schema


CREATE TABLE orders (
	id INTEGER, 
	customer_name TEXT NOT NULL, 
	product_name TEXT NOT NULL, 
	quantity INTEGER NOT NULL, 
	price REAL NOT NULL, 
	total REAL NOT NULL, 
	PRIMARY KEY (id)
)

/*
3 rows from orders table:
id	customer_name	product_name	qu